# Agregate the data from the ROBIN simulator

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import pandas as pd


## Set parameters and load the data

In [3]:
path_kernel_output = '../data/MAD-BCN/kernel_output'
kernel_output_file = 'output_with_service.csv'
path_prices_data = '../data/MAD-BCN/prices/prices_MAD-BCN_2025.csv'
path_agregated_data = '../data/MAD-BCN/aggregated/aggregated_MAD-BCN_2025.csv'
path_final_data = '../data/MAD-BCN/aggregated/MAD-BCN_2025.csv'

figures_path = '../reports/figures/MAD-BCN'

os.makedirs(figures_path, exist_ok=True)


In [4]:
prices_data = pd.read_csv(path_prices_data)
prices_data.head()

,trip_id,origin,destination,tsp,train_type,departure,arrival,duration,service_id,BasicSeat
0,1,60000,71801,AVLO,AVLO,2025-01-01 06:12:00,2025-01-01 08:49:00,0 days 02:37:00,00001_01-01-2025-06.12,32.26
1,2,60000,71801,IRYO,IRYO,2025-01-01 06:22:00,2025-01-01 08:59:00,0 days 02:37:00,00002_01-01-2025-06.22,37.30
2,3,60000,71801,AVE,AVE,2025-01-01 06:27:00,2025-01-01 09:25:00,0 days 02:58:00,00003_01-01-2025-06.27,62.39
3,4,60000,71801,AVE,AVE,2025-01-01 06:57:00,2025-01-01 09:34:00,0 days 02:37:00,00004_01-01-2025-06.57,117.44
4,5,60000,71801,OUIGO,OUIGO,2025-01-01 07:02:00,2025-01-01 09:54:00,0 days 02:52:00,00005_01-01-2025-07.02,47.70


In [5]:
kernel_output = pd.read_csv(os.path.join(path_kernel_output, kernel_output_file))

kernel_output

,id,user_pattern,departure_station,arrival_station,arrival_day,arrival_time,purchase_date,service,service_departure_time,service_arrival_time,seat,price,utility,best_service,best_seat,best_utility,service_id,train_type
0,10788,Leisure,60000,71801,2025-01-01,13.841214,2024-12-06,00020_01-01-2025-20.27,20.450000,23.800000,BasicSeat,16.99,11.774326,00020_01-01-2025-20.27,BasicSeat,11.774326,00020_01-01-2025-20.27,AVE
1,10660,Leisure,60000,71801,2025-01-01,8.487919,2024-12-08,00004_01-01-2025-07.57,7.950000,10.566667,BasicSeat,26.07,11.031351,00004_01-01-2025-07.57,BasicSeat,11.031351,00004_01-01-2025-07.57,AVE
2,124078,Leisure,60000,71801,2025-01-04,8.505233,2024-12-08,00015_04-01-2025-17.17,17.283333,19.900000,BasicSeat,10.00,9.466602,00015_04-01-2025-17.17,BasicSeat,9.466602,00015_04-01-2025-17.17,OUIGO
3,9333,Leisure,60000,71801,2025-01-01,12.059012,2024-12-09,00006_01-01-2025-07.27,7.450000,10.766667,BasicSeat,18.51,9.800671,00006_01-01-2025-07.27,BasicSeat,9.800671,00006_01-01-2025-07.27,AVE
4,10725,Leisure,60000,71801,2025-01-01,14.041513,2024-12-09,00006_01-01-2025-07.27,7.450000,10.766667,BasicSeat,18.51,11.934950,00006_01-01-2025-07.27,BasicSeat,11.934950,00006_01-01-2025-07.27,AVE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8735737,8735725,Business-Morning,60000,71801,2025-12-31,8.322414,2025-12-31,00003_31-12-2025-06.27,6.450000,9.416667,BasicSeat,51.89,1.207246,00003_31-12-2025-06.27,BasicSeat,1.207246,00003_31-12-2025-06.27,AVE
8735738,8735729,Business-Evening,60000,71801,2025-12-31,18.333211,2025-12-31,00004_31-12-2025-14.57,14.950000,17.566667,BasicSeat,35.20,11.894837,00004_31-12-2025-14.57,BasicSeat,11.894837,00004_31-12-2025-14.57,AVE
8735739,8735732,Business-Midday,60000,71801,2025-12-31,15.482992,2025-12-31,00006_31-12-2025-11.27,11.450000,14.766667,BasicSeat,40.93,10.229034,00006_31-12-2025-11.27,BasicSeat,10.229034,00006_31-12-2025-11.27,AVE
8735740,8735734,Business-Morning,60000,71801,2025-12-31,8.059825,2025-12-31,00003_31-12-2025-06.27,6.450000,9.416667,BasicSeat,51.89,2.359374,00003_31-12-2025-06.27,BasicSeat,2.359374,00003_31-12-2025-06.27,AVE


## Aggregated dataset

In [6]:
# Columnas del dataset
# - service_id
# - train_type
# - year
# - month
# - day_of_week
# - departure_time
# - duration
# - price
# - passengers


# Otras columnas que podrían añadirse:
# - Fecha min/max/avg de compra del billete
# - arrival_time
# - Dia entre semana o fin de semana

# - Regional holiday o similar para indicar periodos alta ocupación

In [7]:
# Remove passengers that do not travel, i.e. 'service_id' is NaN
n_rows = kernel_output.shape[0]
kernel_output = kernel_output[kernel_output['service_id'].notna()]

print(f"Number of travelers out of the whole dataset: {kernel_output.shape[0]} / {n_rows}")

Number of travelers out of the whole dataset: 8471506 / 8735742


In [8]:
# Consider that the passengers on 'AVE INT' are equivalent to 'AVE'
kernel_output['train_type'] = kernel_output['train_type'].replace('AVE INT', 'AVE')
prices_data['train_type'] = prices_data['train_type'].replace('AVE INT', 'AVE')
kernel_output['train_type'].unique()

/var/folders/n5/s8_w_cgn3jd6kv8h7psy1jnr0000gn/T/ipykernel_67830/3347484802.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  kernel_output['train_type'] = kernel_output['train_type'].replace('AVE INT', 'AVE')


array(['AVE', 'OUIGO', 'AVLO', 'IRYO'], dtype=object)

In [9]:
# Create the new dataframe
agg_dataset = pd.DataFrame()

agg_dataset['service_id'] = prices_data['service_id']
agg_dataset['train_type'] = prices_data['train_type']

departure_dt = pd.to_datetime(prices_data['departure'], format='%Y-%m-%d %H:%M:%S')
arrival_dt = pd.to_datetime(prices_data['arrival'], format='%Y-%m-%d %H:%M:%S')
agg_dataset['year'] = departure_dt.dt.year
agg_dataset['month'] = departure_dt.dt.month
agg_dataset['day_of_week'] = departure_dt.dt.dayofweek
agg_dataset['departure_time'] = round(departure_dt.dt.hour + departure_dt.dt.minute / 60 + departure_dt.dt.second / 3600, 2)

duration_dt = arrival_dt - departure_dt
agg_dataset['duration'] = duration_dt.dt.total_seconds() / 60

agg_dataset['price'] = prices_data['BasicSeat']

# Count the number of passengers per service_id
kernel_output_grouped = kernel_output.groupby('service_id').size()
agg_dataset['passengers'] = agg_dataset['service_id'].map(kernel_output_grouped).fillna(0)
agg_dataset['passengers'] = agg_dataset['passengers'].astype(int)

print(f"Number of travelers in the agg_dataset: {agg_dataset['passengers'].sum()} / {n_rows}")
print(f"Number of services with passengers in the agg_dataset: {kernel_output['service_id'].nunique()} / {agg_dataset['service_id'].nunique()}")

# Save the agg_dataset
agg_dataset.to_csv(path_agregated_data, index=False)
print(f"agg_dataset saved to {path_agregated_data}")

Number of travelers in the agg_dataset: 8471506 / 8735742
Number of services with passengers in the agg_dataset: 14601 / 15386
agg_dataset saved to ../data/MAD-BCN/aggregated/aggregated_MAD-BCN_2025.csv


In [10]:
agg_dataset

,service_id,train_type,year,month,day_of_week,departure_time,duration,price,passengers
0,00001_01-01-2025-06.12,AVLO,2025,1,2,6.20,157.0,32.26,1068
1,00002_01-01-2025-06.22,IRYO,2025,1,2,6.37,157.0,37.30,600
2,00003_01-01-2025-06.27,AVE,2025,1,2,6.45,178.0,62.39,166
3,00004_01-01-2025-06.57,AVE,2025,1,2,6.95,157.0,117.44,0
4,00005_01-01-2025-07.02,OUIGO,2025,1,2,7.03,172.0,47.70,17
...,...,...,...,...,...,...,...,...,...
15381,00004_31-12-2025-20.05,AVE,2025,12,2,20.08,157.0,45.42,113
15382,00009_31-12-2025-20.22,IRYO,2025,12,2,20.37,172.0,58.12,0
15383,00020_31-12-2025-20.27,AVE,2025,12,2,20.45,201.0,36.27,36
15384,00005_31-12-2025-21.02,OUIGO,2025,12,2,21.03,172.0,40.84,0


## Final dataset

In [11]:
# Mismo dataset añadiendo las columnas para cada servicio (2 antes y 2 después) que compite:
# - distancia temporal (negativa si es antes)
# - Precio de la competencia
# - Compañia
# - Duración del viaje


In [12]:

final_dataset = agg_dataset.copy()
# Add the columns for each service that competes (2 before and 2 after):
for i in range(-2, 3):
    if i == 0:
        continue
    # Get the service_id of the service that competes
    #final_dataset[f'service_id_competitor_{i}'] = final_dataset['service_id'].shift(-i)

    # Get the temporal distance (negative if it is before)
    time_distance = final_dataset['departure_time'].shift(-i) - final_dataset['departure_time']
    if i < 0:
        # Means that the service is before, so if the distance is positive, means the train was the day before
        time_distance[time_distance > 0] = -time_distance[time_distance > 0]
    elif i > 0:
        # Means that the service is after, so if the distance is negative, means the train was the day after
        time_distance[time_distance < 0] = -time_distance[time_distance < 0]
    final_dataset[f'time_distance_competitor_{i}'] = round(time_distance * 60, 2)

    # Get the price of the competitor
    final_dataset[f'price_competitor_{i}'] = final_dataset['price'].shift(-i)

    # Get the train type of the competitor
    final_dataset[f'train_type_competitor_{i}'] = final_dataset['train_type'].shift(-i)

    # Get the duration of the competitor
    final_dataset[f'duration_competitor_{i}'] = final_dataset['duration'].shift(-i)


# Count and remove the rows with NaN values
n_rows = final_dataset.shape[0]
final_dataset = final_dataset.dropna()
print(f"There have been {n_rows - final_dataset.shape[0]} rows with NaN values removed")

# Save the final dataset
final_dataset.to_csv(path_final_data, index=False)
print(f"final_dataset saved to {path_final_data}")


There have been 4 rows with NaN values removed
final_dataset saved to ../data/MAD-BCN/aggregated/MAD-BCN_2025.csv


In [13]:
final_dataset

,service_id,train_type,year,month,day_of_week,departure_time,duration,price,passengers,time_distance_competitor_-2,...,train_type_competitor_-1,duration_competitor_-1,time_distance_competitor_1,price_competitor_1,train_type_competitor_1,duration_competitor_1,time_distance_competitor_2,price_competitor_2,train_type_competitor_2,duration_competitor_2
2,00003_01-01-2025-06.27,AVE,2025,1,2,6.45,178.0,62.39,166,-15.0,...,IRYO,157.0,30.0,117.44,AVE,157.0,34.8,47.70,OUIGO,172.0
3,00004_01-01-2025-06.57,AVE,2025,1,2,6.95,157.0,117.44,0,-34.8,...,AVE,178.0,4.8,47.70,OUIGO,172.0,25.2,34.55,IRYO,157.0
4,00005_01-01-2025-07.02,OUIGO,2025,1,2,7.03,172.0,47.70,17,-34.8,...,AVE,157.0,20.4,34.55,IRYO,157.0,25.2,18.51,AVE,199.0
5,00002_01-01-2025-07.22,IRYO,2025,1,2,7.37,157.0,34.55,227,-25.2,...,OUIGO,172.0,4.8,18.51,AVE,199.0,34.8,26.07,AVE,157.0
6,00006_01-01-2025-07.27,AVE,2025,1,2,7.45,199.0,18.51,14405,-25.2,...,IRYO,157.0,30.0,26.07,AVE,157.0,60.0,94.98,AVE,172.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15379,00009_31-12-2025-19.29,IRYO,2025,12,2,19.48,172.0,25.91,1946,-57.0,...,AVE,157.0,5.4,29.22,AVLO,201.0,36.0,45.42,AVE,157.0
15380,00019_31-12-2025-19.34,AVLO,2025,12,2,19.57,201.0,29.22,181,-37.2,...,IRYO,172.0,30.6,45.42,AVE,157.0,48.0,58.12,IRYO,172.0
15381,00004_31-12-2025-20.05,AVE,2025,12,2,20.08,157.0,45.42,113,-36.0,...,AVLO,201.0,17.4,58.12,IRYO,172.0,22.2,36.27,AVE,201.0
15382,00009_31-12-2025-20.22,IRYO,2025,12,2,20.37,172.0,58.12,0,-48.0,...,AVE,157.0,4.8,36.27,AVE,201.0,39.6,40.84,OUIGO,172.0
